# Controlled peer donors v1
Read docs/controlled_peer_donors_colab.md first. GPU stages default off; no new scientific outcome is claimed.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys, json
from google.colab import drive
GIT_REF = "PASTE_FULL_SHA_OF_YOUR_REVIEWED_PUSH"
REPO_URL = "https://github.com/Soqoro/Pact.git"
assert re.fullmatch(r"[0-9a-fA-F]{40}", GIT_REF)
drive.mount("/content/drive")
SCRATCH = Path("/content/pact-scratch")
CHECKOUT = SCRATCH / "checkout"
SCRATCH.mkdir(parents=True, exist_ok=True)
def git(*args):
    return subprocess.check_output(["git", *args], text=True).strip()
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(CHECKOUT)], check=True)
assert not git("-C", str(CHECKOUT), "status", "--porcelain"), "Preserve local checkout edits first."
assert git("-C", str(CHECKOUT), "remote", "get-url", "origin") == REPO_URL
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", GIT_REF], check=True)
assert git("-C", str(CHECKOUT), "rev-parse", "HEAD").lower() == GIT_REF.lower()
os.chdir(CHECKOUT)
sys.path.insert(0, str(CHECKOUT / "src"))
os.environ["HF_HOME"] = str(SCRATCH / "cache/huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
from pact.colab import install_dependencies
install_dependencies(CHECKOUT)
# Focused CPU checks; these do not download model weights.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests",
                "-p", "test_controlled_donors.py", "-v"], check=True)


In [ ]:
from pact.storage import storage_operation
from pact.training.preparation_restore import restore_preparation_references
from pact.training.controlled_donors import load_config as load_rx_config
from pact.training.controlled_donors import plan as controlled_plan, PARENT_SHA
from pact.training.receiver_study import initial_recipe
from pact.environment import runtime_fingerprint
from pact.util import file_hash
DRIVE = Path("/content/drive/MyDrive/PACT")
PREP = DRIVE / "actor-preparation/qwen3-preparation-120-001"
CONFIG = CHECKOUT / "experiments/controlled_peer_donors_v1.json"
SELECTION = CHECKOUT / "experiments/receiver_supervision_selection.json"
config = load_rx_config(CONFIG)
TRAIN_ZIP = SCRATCH / "sources/preparation-training.zip"
TRAIN_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not TRAIN_ZIP.exists():
    print(storage_operation("bundle-restore",
        PREP / "training/bundles/qwen3-preparation-120-001-handoff-1789930640697369191.zip",
        TRAIN_ZIP, sha256=config.initialization_bundle_sha256, timeout_seconds=600))
assert file_hash(TRAIN_ZIP) == config.initialization_bundle_sha256
INITIAL = SCRATCH / "actor-preparation/qwen3-preparation-120-001-inference"
if not INITIAL.exists():
    print(restore_preparation_references(
        PREP / "training/snapshots/1789930640656395583-479df47c14d9",
        INITIAL, timeout_seconds=600))
DATA = SCRATCH / "training-data/proposal-1200"
if not DATA.exists():
    subprocess.run([sys.executable, "-m", "pact", "prepare-training-data",
        "--cache-dir", str(SCRATCH / "cache/datasets"), "--output-dir", str(DATA),
        "--items", "1200", "--seed", "20260918", "--download"], check=True)
PARENT_ZIP = SCRATCH / "sources/receiver-parent.zip"
if not PARENT_ZIP.exists():
    print(storage_operation("bundle-restore",
        DRIVE / "receiver-supervision/qwen3-receiver-supervision-001/bundles/qwen3-receiver-supervision-001-handoff-1790063068160794241.zip",
        PARENT_ZIP, sha256=PARENT_SHA, timeout_seconds=600))
assert file_hash(PARENT_ZIP) == PARENT_SHA
ACQUISITION_ID = "qwen3-controlled-peer-donors-001"
PLAN = controlled_plan(config, DATA, TRAIN_ZIP, SELECTION, PARENT_ZIP, ACQUISITION_ID)
initial_recipe(PLAN, INITIAL)  # checks actual adapter files; no model load
print("Budget:", PLAN["budget"])
print("Runtime:", runtime_fingerprint())
assert runtime_fingerprint() == PLAN["model"]["runtime_fingerprint"], "Runtime changed; return this output for review."


In [ ]:
RUN_ID = "qwen3-receiver-supervision-controlled-001"
RUN = SCRATCH / "receiver-supervision" / RUN_ID
PERSISTENT = DRIVE / "receiver-supervision" / RUN_ID
COMMON = ["--config", str(CONFIG), "--data-dir", str(DATA),
    "--initialization-bundle", str(TRAIN_ZIP), "--initialization-root", str(INITIAL),
    "--selection", str(SELECTION), "--run-dir", str(RUN),
    "--persistent", str(PERSISTENT), "--cache-dir", str(SCRATCH / "cache"),
    "--parent-bundle", str(PARENT_ZIP), "--acquisition-id", ACQUISITION_ID]
def study(stage, *options):
    subprocess.run([sys.executable, "-m", "pact.training.controlled_donors",
                    stage, *COMMON, *options], check=True)
study("plan")  # dry run, no model load or generation
if not RUN.exists():
    snapshots = sorted((PERSISTENT / "snapshots").glob("*"))
    if snapshots:
        study("restore", "--snapshot", str(snapshots[-1]))
    else:
        study("plan", "--execute")  # freezes membership and verifies persistence


In [ ]:
from pact.util import read_json
RUN_ACQUISITION = False  # set True after reviewing the printed plan
if RUN_ACQUISITION:
    study("acquire", "--execute")
if (RUN / "contexts.json").exists():
    CONTEXTS = read_json(RUN / "contexts.json")
    print({p: {k: v for k, v in part.items() if k != "records"}
           for p, part in CONTEXTS["partitions"].items()})
    if CONTEXTS["status"] != "ready":
        study("report")
        raise RuntimeError("insufficient_context_support: return the handoff ZIP; no more draws or training.")


In [ ]:
RUN_TRAINING = False  # set True only after the context gate passes
if RUN_TRAINING:
    assert read_json(RUN / "contexts.json")["status"] == "ready"
for arm in (("task_sft", "receiver_sft") if RUN_TRAINING else ()):
    arm_root = RUN / "training" / arm
    options = ["--arm", arm, "--execute"]
    if (arm_root / "run.json").exists():
        options.append("--resume")
    study("train", *options)
    assert read_json(arm_root / "status.json")["complete"]


In [ ]:
RUN_EVALUATION = False  # set True only after BOTH trained arms finish
for arm in (("frozen", "task_sft", "receiver_sft") if RUN_EVALUATION else ()):
    study("evaluate", "--arm", arm, "--execute")


In [ ]:
# This also works for an acquisition support stop; report.json exists only after evaluation.
if all((RUN / "evaluation" / a / "status.json").exists()
       for a in ("frozen", "task_sft", "receiver_sft")):
    study("report")
    REPORT = read_json(RUN / "report.json")
    print(REPORT["generation_accounting"])
    print(REPORT["controlled_donor_comparisons"])
else:
    study("export")
    print(read_json(RUN / "source_report.json"))
from google.colab import files
latest = max((RUN.parent / "bundles").glob(f"{RUN_ID}-handoff-*.zip"), key=lambda p: p.stat().st_mtime_ns)
print("Review:", latest, "SHA256:", file_hash(latest))
print("Drive copy:", PERSISTENT / "bundles" / latest.name)
files.download(str(latest))
